In [ ]:
"""
Demo notebook for Multi-Agent Customer Support Platform
This notebook demonstrates how to use the system
"""

# Cell 1: Imports and setup
import sys
sys.path.insert(0, '/e:/GUVI_projects/Multi-Agent Customer Support Intelligence Platform')

from backend.agents.orchestrator import AgentOrchestrator
from backend.rag import RAGPipeline
from backend.config import get_settings
import json


# Cell 2: Initialize the system
print("Initializing Multi-Agent Customer Support Platform...")

settings = get_settings()
rag_pipeline = RAGPipeline(settings.faiss_index_path)
orchestrator = AgentOrchestrator(rag_pipeline)

print("✅ System initialized")


# Cell 3: Sample tickets
sample_tickets = [
    {
        "text": "My order ORD-123456 hasn't arrived yet. It was supposed to come 2 days ago.",
        "name": "John Smith",
        "email": "john.smith@example.com"
    },
    {
        "text": "I received the wrong item. I ordered a blue shirt but got a red one. Very disappointed.",
        "name": "Sarah Johnson",
        "email": "sarah.j@example.com"
    },
    {
        "text": "Your website charged me twice for the same order! Please refund immediately.",
        "name": "Michael Chen",
        "email": "m.chen@example.com"
    }
]

print(f"Sample tickets prepared: {len(sample_tickets)}")


# Cell 4: Process a ticket
print("Processing ticket 1...")

result = orchestrator.process_ticket(
    ticket_text=sample_tickets[0]["text"],
    customer_name=sample_tickets[0]["name"],
    customer_email=sample_tickets[0]["email"]
)

print(json.dumps({
    "status": result["status"],
    "category": result["category"],
    "priority": result["priority"],
    "sentiment": result["sentiment"],
    "requires_escalation": result["requires_escalation"],
    "assigned_team": result["assigned_team"]
}, indent=2))


# Cell 5: Display generated response
print("\nGenerated Response:")
print("-" * 60)
print(result["final_response"])
print("-" * 60)


# Cell 6: View agent logs
print("\n📊 Agent Execution Summary:")
for agent_name, agent_result in result["agent_results"].items():
    status = "✅" if agent_result.get("status") == "success" else "❌"
    exec_time = agent_result.get("execution_time_ms", 0)
    print(f"{status} {agent_result.get('agent_name')}: {exec_time:.1f}ms")

print(f"\nTotal execution time: {result['total_execution_time_ms']:.1f}ms")


# Cell 7: Batch process multiple tickets
print("\n\nBatch Processing All Tickets...")
results = []

for i, ticket in enumerate(sample_tickets, 1):
    print(f"\nProcessing ticket {i}/{len(sample_tickets)}...")
    
    result = orchestrator.process_ticket(
        ticket_text=ticket["text"],
        customer_name=ticket["name"],
        customer_email=ticket["email"]
    )
    
    results.append({
        "customer": ticket["name"],
        "category": result.get("category"),
        "priority": result.get("priority"),
        "sentiment": result.get("sentiment"),
        "escalated": result.get("requires_escalation"),
        "exec_time_ms": result.get("total_execution_time_ms")
    })
    
    print(f"  ✓ Processed - Category: {result.get('category')}, Priority: {result.get('priority')}")


# Cell 8: Summary statistics
print("\n\n📈 Batch Processing Summary:")
print(f"Total tickets: {len(results)}")
print(f"Escalated: {sum(1 for r in results if r['escalated'])}")
print(f"Auto-resolved: {sum(1 for r in results if not r['escalated'])}")
print(f"Average execution time: {sum(r['exec_time_ms'] for r in results) / len(results):.1f}ms")

print("\nCategory distribution:")
from collections import Counter
categories = Counter(r['category'] for r in results)
for cat, count in categories.most_common():
    print(f"  {cat}: {count}")

print("\nPriority distribution:")
priorities = Counter(r['priority'] for r in results)
for pri, count in priorities.most_common():
    print(f"  {pri}: {count}")


# Cell 9: Demo complete
print("\n" + "="*60)
print("✅ Demo completed successfully!")
print("\nNext steps:")
print("1. Run the FastAPI backend: python backend/main.py")
print("2. Start the Streamlit frontend: streamlit run frontend/app.py")
print("3. Open your browser to http://localhost:8501")
print("="*60)